# 05. AI vs 규칙 기반: 어느 쪽을 쓸지 판정

"왜 규칙 대신 AI를 썼는가"를 숫자로 답하는 노트북입니다.

1. **결과를 보기 전에** 판정 기준을 정합니다.
2. 같은 시험을 **여러 번 반복**합니다. 여러 해 데이터면 연도별로 5번, 7일 샘플이면 지역 블록을 바꿔 5번입니다.
3. AI를 **가장 강한 규칙 모델**과 비교합니다. 대륙 정보까지 준 규칙 모델도 포함해서, 규칙 쪽이 정보 부족으로 지는 일이 없게 합니다.
4. 정한 기준대로 판정합니다.

## ① 판정 기준 (결과를 보기 전에 고정)

| 기준 | 뜻 |
| --- | --- |
| `min_improvement` | 주 지표(**합계 MAE**)를 최강 규칙 모델보다 평균 몇 % 이상 줄여야 하는가 |
| `min_win_share` | 전체 fold 중 몇 % 이상에서 AI가 이겨야 하는가 |
| `max_extra_bias` | AI의 평균 \|편향\|이 규칙 모델보다 몇 %p까지 커져도 되는가 |

**합계 MAE**는 10° 블록 × 기간별 MODIS 합계의 평균 절대 오차입니다. 달력이 실제로 보여주는 값에 가장 가까운 지표라 주 지표로 정했습니다.
기준을 바꾸려면 **결과를 보기 전에** 이 셀에서 바꾸고, 바꾼 이유를 메모로 남기세요.

In [ ]:
from fsml.compare import Criteria

CRITERIA = Criteria(min_improvement=0.10, min_win_share=0.8, max_extra_bias=5.0)
CRITERIA

## ② 데이터와 반복 시험 구성

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from fsml import evaluate  # 한글 폰트 설정 포함
from fsml.compare import PRIMARY, cross_validate, decide, summarize
from fsml.features import make_features, training_rows
from fsml.paths import default_grid
from fsml.splits import auto_folds

GRID = default_grid()
grid = pd.read_parquet(GRID)
res = float(np.round(np.diff(np.sort(grid["lat"].unique())).min(), 6))  # 격자 크기(°)
data, FEATURES = make_features(training_rows(grid), res)
kind, folds = auto_folds(data, n=5)
print(f"{GRID.name} | 검증 방식: {'연도별' if kind == 'time' else '지역 블록별'} {len(folds)}회")
for k, (tr, va, te) in enumerate(folds, start=1):
    print(f"  fold {k}: 학습 {tr.sum():,} / 검증 {va.sum():,} / 시험 {te.sum():,}")

## ③ 반복 시험 결과

In [ ]:
results = cross_validate(data, FEATURES, folds)
results.round(3)

In [ ]:
summarize(results).round(3)

In [ ]:
wide = results.pivot(index="fold", columns="모델", values=PRIMARY)
ax = wide.plot.bar(figsize=(10, 4), rot=0)
ax.set_ylabel(f"{PRIMARY} (낮을수록 좋음)")
ax.set_title("fold별 합계 오차");

## ④ 판정

In [ ]:
verdict, detail = decide(results, CRITERIA)
print(verdict)
pd.Series(detail)

## 메모

- 7일 샘플이면 이 판정은 **연습용**입니다. 실제 과거 데이터로 격자를 만든 뒤 다시 실행하면, 자동으로 연도별 검증(최근 5개 연도)으로 바뀝니다.
- 여러 해 데이터가 생기면 추가할 지표 (TODO)
  - 화재 시즌 정점 월이 실제 MODIS와 맞는 비율
  - 평년보다 심한 달(이상)을 실제 MODIS와 똑같이 잡는 비율
- 발표에는 ① 기준표, ③ fold별 막대그래프, ④ 판정 결과를 그대로 쓰면 됩니다.